In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
print('rakesh')

rakesh


In [3]:
import importlib

import  src.recommenders.item_cf as pipeline

importlib.reload(pipeline)

<module 'src.recommenders.item_cf' from 'C:\\Users\\rauni\\Documents\\Learning\\Projects\\ads-intelligence-platform\\src\\recommenders\\item_cf.py'>

In [4]:
from src.pipelines.build_master_dataset import build_master_dataset
from src.recommenders.item_cf import ItemBasedCollaborativeFiltering

master = build_master_dataset(save=False)

master.head()



,user_id,video_id,date,hourmin,time_ms,is_click,is_like,is_follow,is_comment,is_forward,...,video_type,upload_dt,upload_type,visible_status,video_duration,server_width,server_height,music_id,music_type,tag
0,0,1527,20220411,1900,1649675512388,0,0,0,0,0,...,NORMAL,2022-04-10,Web,0.0,209900.0,1172.0,720.0,9154013497,9.0,39
1,0,7405,20220416,2000,1650111976017,0,0,0,0,0,...,NORMAL,2022-04-09,Kmovie,0.0,65400.0,720.0,1280.0,9145692691,9.0,12
2,0,6026,20220420,1600,1650444367095,0,0,0,0,0,...,NORMAL,2022-04-10,Web,0.0,170833.0,720.0,1280.0,9156647291,9.0,39
3,1,6354,20220411,1100,1649645295928,0,0,0,0,0,...,NORMAL,2022-04-10,Web,0.0,255160.0,1280.0,720.0,9154256938,9.0,3
4,1,3645,20220411,1100,1649648827559,0,0,0,0,0,...,NORMAL,2022-04-10,Web,0.0,79733.0,720.0,1280.0,9152899551,9.0,22


In [5]:
model = ItemBasedCollaborativeFiltering()

In [6]:
interaction_matrix = model.build_interaction_matrix(master)

In [7]:
interaction_matrix.shape

(26210, 7538)

In [8]:

interaction_matrix.head()

video_id,0,1,2,3,4,5,6,7,8,9,...,7573,7574,7575,7576,7577,7578,7579,7580,7581,7582
user_id,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:

interaction_matrix.iloc[:10, :10]

video_id,0,1,2,3,4,5,6,7,8,9
user_id,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0


In [10]:

### How many videos does an average user interact with?

interaction_matrix.sum(axis=1).describe()

count    26210.000000
mean        41.692102
std         42.138149
min          1.000000
25%         13.000000
50%         29.000000
75%         57.000000
max        729.000000
dtype: float64

In [11]:
### How many users interact with an average video?

interaction_matrix.sum(axis=0).describe()

count    7538.000000
mean      144.965508
std       351.902023
min         1.000000
25%        15.000000
50%        42.000000
75%       130.000000
max      8961.000000
dtype: float64

In [12]:
interaction_matrix.values.sum()

np.int64(1092750)

In [13]:
interaction_matrix.astype(bool).sum().sum()

np.int64(1092750)

In [14]:
interaction_matrix.head(20)

video_id,0,1,2,3,4,5,6,7,8,9,...,7573,7574,7575,7576,7577,7578,7579,7580,7581,7582
user_id,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Interaction Matrix Analysis

The interaction matrix transforms the raw interaction log into a **User × Video** matrix, where:

- Each row represents a user.
- Each column represents a video.
- A value of **1** indicates that the user has interacted with the video.
- A value of **0** indicates no observed interaction.

This representation is the foundation of classical collaborative filtering algorithms.

---

# Dataset Summary

| Metric | Value |
|---------|------:|
| Users | 26,210 |
| Videos | 7,538 |
| Total User-Video Interactions | 1,092,750 |
| Average Videos per User | **43** |
| Median Videos per User | **29** |
| Average Users per Video | **145** |
| Median Users per Video | **42** |

---

# Key Findings

## 1. User activity is highly uneven

On average, a user interacts with **43 videos**, while the median is only **29**.

Since the mean is noticeably higher than the median, user activity is **right-skewed**.

This indicates that:

- Most users interact with a relatively small number of videos.
- A small number of highly active users consume significantly more content than the average user.

This is a common characteristic of real-world recommendation systems.

---

## 2. Video popularity is also uneven

An average video is watched by approximately **145 users**, while the median video is watched by only **42 users**.

Again, the mean is much larger than the median.

This confirms that:

- A small number of videos receive a disproportionately large amount of user attention.
- Most videos belong to the long tail and receive comparatively fewer interactions.

---

## 3. Duplicate interactions were removed

The interaction matrix contains **1,092,750 unique user-video interactions**.

Although the original interaction log contains over **1.14 million events**, repeated interactions between the same user and video are collapsed into a single interaction.

This is intentional because Item-Based Collaborative Filtering models **whether a user interacted with an item**, rather than **how many times** the interaction occurred.

For example:

| User | Video | Raw Log |
|------|-------|---------|
| U1 | Iron Man | 5 interactions |

becomes

| User | Video |
|------|-------|
| U1 | Iron Man |

This binary representation simplifies similarity computation and is the standard approach in classical collaborative filtering.

---

# Why is the Interaction Matrix Important?

Once the interaction matrix is built, every video becomes a vector representing the users who interacted with it.

Instead of asking:

> "How popular is this video?"

we can now ask:

> **"Which videos are watched by similar groups of users?"**

This forms the basis of **Item-Based Collaborative Filtering**.

The next step is therefore to compute the **similarity between video vectors**, allowing us to recommend videos that are frequently consumed together.

## Items Similarity

In [15]:
similarity_matrix = model.compute_item_similarity(
    interaction_matrix
)

In [16]:
similarity_matrix.shape

(7538, 7538)

In [17]:
similarity_matrix.iloc[:5, :5]

video_id,0,1,2,3,4
video_id,,,,,
0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,0.0,1.0


In [18]:
similarity_matrix.head()

video_id,0,1,2,3,4,5,6,7,8,9,...,7573,7574,7575,7576,7577,7578,7579,7580,7581,7582
video_id,,,,,,,,,,,,,,,,,,,,,
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.06455,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.058926,0.094491,0.0,0.000000
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.080322,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.051848
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.00000,0.009577,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000


In [19]:
similarity_matrix.loc[6921].sort_values(
    ascending=False
).head(10)

video_id
6921    1.000000
2303    0.475227
3310    0.441660
6714    0.383898
5391    0.370273
3273    0.369140
7136    0.365289
6989    0.361740
4644    0.361711
1725    0.355126
Name: 6921, dtype: float64

In [20]:
similarity_matrix.loc[2303, 6921]

np.float64(0.4752274879329156)

In [21]:
similar_videos = model.get_similar_videos(
    video_id=6921,
    k=10
)

similar_videos

,rank,video_id,similarity
0,1,2303,0.475227
1,2,3310,0.441660
2,3,6714,0.383898
3,4,5391,0.370273
4,5,3273,0.369140
5,6,7136,0.365289
6,7,6989,0.361740
7,8,4644,0.361711
8,9,1725,0.355126
9,10,3955,0.349676


In [22]:
master["video_id"].sample(1).iloc[0]

np.int64(4479)

In [23]:
model.get_similar_videos(
    video_id=6822,
    k=10
)



,rank,video_id,similarity
0,1,2072,0.331872
1,2,2243,0.260267
2,3,2349,0.223683
3,4,3573,0.222998
4,5,165,0.219054
5,6,5597,0.210507
6,7,2225,0.202637
7,8,5037,0.197094
8,9,3271,0.192762
9,10,5736,0.182199


In [24]:
model.get_similar_videos(
    video_id=6921,
    k=100000
)

,rank,video_id,similarity
0,1,2303,0.475227
1,2,3310,0.441660
2,3,6714,0.383898
3,4,5391,0.370273
4,5,3273,0.369140
...,...,...,...
7532,7533,3525,0.000000
7533,7534,6757,0.000000
7534,7535,113,0.000000
7535,7536,110,0.000000


In [25]:
from importlib import reload
import src.recommenders.item_cf

reload(src.recommenders.item_cf)

from src.recommenders.item_cf import ItemBasedCollaborativeFiltering

In [26]:
user_id = master["user_id"].sample(1).iloc[0]

user_id

np.int64(17424)

In [27]:
watched = (
    master.loc[
        master.user_id == user_id,
        "video_id"
    ]
    .drop_duplicates()
    .tolist()
)

len(watched)

102

In [28]:
watched 

[681,
 1251,
 4786,
 2146,
 5563,
 2026,
 5940,
 6984,
 3207,
 6901,
 920,
 2098,
 6139,
 6565,
 6510,
 4345,
 4590,
 2087,
 2580,
 2063,
 605,
 2964,
 5075,
 3283,
 5877,
 3796,
 28,
 2407,
 4550,
 4958,
 7498,
 7006,
 476,
 6205,
 7243,
 5848,
 333,
 943,
 2609,
 4966,
 2464,
 2368,
 1502,
 6774,
 647,
 6687,
 4636,
 5510,
 3685,
 1696,
 2245,
 549,
 791,
 6370,
 3000,
 3551,
 841,
 913,
 3587,
 1590,
 7002,
 1056,
 3307,
 7128,
 2552,
 961,
 4435,
 4990,
 537,
 172,
 2521,
 1783,
 5796,
 1487,
 5931,
 2583,
 1200,
 3539,
 3354,
 76,
 526,
 2753,
 2189,
 2379,
 2103,
 5522,
 6261,
 6308,
 2990,
 4595,
 2639,
 6697,
 7299,
 2347,
 5486,
 4670,
 2926,
 3836,
 53,
 3139,
 4732,
 6847]

In [29]:
recommendations = model.recommend_for_user(
    user_id=user_id,
    interactions=master,
    n=20,
    k=20
)

recommendations

,rank,video_id,similarity_score
0,1,2303,5.325654
1,2,4631,4.464621
2,3,6694,4.172821
3,4,2076,4.024334
4,5,4662,3.859277
5,6,3831,3.553039
6,7,1639,3.160502
7,8,597,2.868719
8,9,2712,2.854551
9,10,3104,2.733252


In [30]:
set(watched).intersection(
    recommendations.video_id
)

set()

In [ ]:
from src.recommenders.popularity import PopularityRecommender

popularity_model = PopularityRecommender()


In [33]:
from src.recommenders.popularity import PopularityRecommender

popularity_model = PopularityRecommender()

popularity_model.fit(master)

popularity_model.recommend_unseen(
    user_id=user_id,
    interactions=master,
    n=20,
)

,rank,video_id,interaction_count
0,1,6921,9110
1,2,2303,7914
2,3,3310,7359
3,4,1868,6406
4,5,7136,5793
5,6,6714,3789
6,7,3273,3771
7,8,7177,3726
8,9,4955,3701
9,10,4644,3631


# Item-Based Collaborative Filtering Evaluation

## Business Question

How are Item-Based Collaborative Filtering recommendations different from a popularity-based recommender?

---

## Recommendation Comparison

### Popularity Recommender

The popularity recommender recommends the globally most interacted videos.

Example:

| Rank | Video | Interaction Count |
|------|--------|------------------:|
|1|6921|9110|
|2|2303|7914|
|3|3310|7359|
|...|...|...|

Every user receives almost the same recommendations (except videos already watched in V2).

---

### Item-Based Collaborative Filtering

Item-Based Collaborative Filtering recommends videos that are similar to the videos the user has already interacted with.

Example:

| Rank | Video | Similarity Score |
|------|--------|-----------------:|
|1|2303|5.33|
|2|4631|4.46|
|3|6694|4.17|
|...|...|...|

Unlike popularity, these recommendations depend entirely on the user's viewing history.

---

# Key Observations

## 1. Recommendations are personalized

Popularity recommends the same globally popular videos to almost every user.

ItemCF generates different recommendations for different users because recommendations are derived from each user's interaction history.

This is the first fully behavior-driven recommender built in this project.

---

## 2. Scores have different meanings

Popularity uses:

> Interaction Count

Higher values simply indicate that a video was watched by many users.

ItemCF uses:

> Similarity Score

The similarity score is the sum of similarity values contributed by every video the user has already watched.

A higher score indicates stronger evidence that the recommended video is related to the user's historical interests.

---

## 3. Recommendations are no longer driven by popularity alone

Although some popular videos may still appear, many recommendations are videos that are highly related to the user's viewing behavior rather than simply being globally popular.

This allows the recommender to surface more relevant content for each individual user.

---

# Business Impact

Compared with a popularity-based recommender, Item-Based Collaborative Filtering provides:

- Personalized recommendations.
- Better content relevance.
- Improved content discovery.
- Greater diversity across users.
- Recommendations based on behavioral similarity instead of global popularity.

---

# Remaining Limitations

Item-Based Collaborative Filtering still has several important limitations.

- New users cannot receive personalized recommendations because they have no interaction history (User Cold Start).
- New videos cannot be recommended because they have no interaction history (Item Cold Start).
- Similarity computation becomes increasingly expensive as the catalog grows.
- Only historical interactions are considered; content information such as title, genre, description, or embeddings is ignored.

---

# Conclusion

Item-Based Collaborative Filtering represents a significant improvement over popularity-based recommendation.

Instead of recommending what is popular, it recommends what is **most similar to the user's historical preferences**.

This marks the transition from heuristic recommendation methods to behavior-based recommendation systems.